# LIBEQG — Fase C: prevalidación local en CPU

Notebook reproducible para validar el testbench antes de consumir créditos de una QPU real. Ejecuta Qiskit Aer y NVIDIA CUDA-Q en CPU, con semillas fijas, circuitos Bell/GHZ-3 e invariantes de *ground truth*. D-Wave Leap queda desactivado por defecto y no se llama desde este notebook.

**Importante:** las semillas hacen repetibles las muestras cuando el backend respeta esa API; no hacen que dos simuladores distintos tengan los mismos conteos. Registra versiones y backend junto a cada resultado.

In [ ]:
from pathlib import Path
import os, sys, json, platform, subprocess, importlib.util

# Localiza la raíz real del repositorio (pyproject.toml), incluso si el notebook
# se abrió desde una subcarpeta. No crea rutas de proyecto falsas.
candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in candidates if (p / 'pyproject.toml').is_file() and (p / 'src' / 'libeqg').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('No encuentro la raíz de LIBEQG. Abre este notebook desde un checkout que contenga pyproject.toml y src/libeqg/.')
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))
print('Raíz:', REPO_ROOT)
print('Python:', platform.python_version(), '| Plataforma:', platform.platform())
print('CUDA-Q CPU target se seleccionará explícitamente en su celda.')


## 1. Instalar dependencias de la Fase C

Ejecuta esta celda en un entorno Python compatible con ambos paquetes. CUDA-Q puede no ofrecer ruedas para todas las combinaciones de sistema operativo/versión de Python; si la instalación falla, no marques CUDA-Q como validado. La instalación necesita acceso al índice de paquetes.

In [ ]:
# Instala desde la raíz del repositorio: dependencias del paquete + ambos simuladores.
# Para un entorno ya preparado, puedes omitir esta celda.
INSTALL_SIMULATORS = True
if INSTALL_SIMULATORS:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-e',
        '.[phase-c-aer,phase-c-cudaq]'
    ])
print('Instalación solicitada. Reinicia el kernel si el entorno te lo pide y vuelve a ejecutar desde el inicio.')

## 2. Configuración determinista y criterio de aceptación

Ground truth ideal: Bell sólo debe producir `00`/`11`; GHZ-3 sólo `000`/`111`. No se compara igualdad exacta de conteos entre Aer y CUDA-Q. Un resultado que contenga estados imposibles en modo ideal falla la prevalidación.

In [ ]:
import hashlib, time
BASE_SEED = 20261016
SHOTS = 2048
CIRCUITS = ('bell', 'ghz3')

def seed_for(simulator_id, circuit_id, base_seed=BASE_SEED):
    payload = f'LIBEQG-FaseC-v1|{base_seed}|{simulator_id}|{circuit_id}'.encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], 'big')

def normalize_counts(counts):
    return {str(k).replace(' ', ''): int(v) for k, v in counts.items() if int(v) > 0}

def validate_ideal_counts(counts, circuit_id):
    allowed = {'bell': {'00', '11'}, 'ghz3': {'000', '111'}}[circuit_id]
    counts = normalize_counts(counts)
    return bool(counts) and all(k in allowed for k in counts)

print('Semillas:', {s: {c: seed_for(s, c) for c in CIRCUITS} for s in ('qiskit_aer', 'cudaq_sim')})

## 3. Qiskit Aer en CPU

Se usa `AerSimulator(method='statevector', device='CPU')` y se repite cada circuito con la misma semilla para comprobar la reproducibilidad local.

In [ ]:
import qiskit, qiskit_aer
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

aer = AerSimulator(method='statevector', device='CPU')
def qiskit_circuit(circuit_id):
    n = 2 if circuit_id == 'bell' else 3
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(1, n):
        qc.cx(0, i)
    qc.measure_all()
    return qc

aer_results = {}
for circuit_id in CIRCUITS:
    circuit = qiskit_circuit(circuit_id)
    seed = seed_for('qiskit_aer', circuit_id)
    counts_a = normalize_counts(aer.run(circuit, shots=SHOTS, seed_simulator=seed).result().get_counts())
    counts_b = normalize_counts(aer.run(circuit, shots=SHOTS, seed_simulator=seed).result().get_counts())
    assert counts_a == counts_b, f'Aer no fue reproducible para {circuit_id}'
    assert validate_ideal_counts(counts_a, circuit_id), f'Aer viola ground truth para {circuit_id}: {counts_a}'
    assert sum(counts_a.values()) == SHOTS
    aer_results[circuit_id] = {'seed': seed, 'shots': SHOTS, 'counts': counts_a, 'repeatable': True, 'ground_truth': 'PASS'}
print('Qiskit:', qiskit.__version__, '| Aer:', qiskit_aer.__version__)
print(json.dumps(aer_results, indent=2))

## 4. NVIDIA CUDA-Q explícitamente en CPU

Se selecciona el target `qpp-cpu` para no depender de una GPU. Si la versión instalada no incluye ese target, la celda debe fallar y la prueba queda como no validada (no se cambia silenciosamente a GPU).

In [ ]:
import cudaq
cudaq.set_target('qpp-cpu')
print('CUDA-Q version:', getattr(cudaq, '__version__', 'versión no expuesta'))
print('Target solicitado: qpp-cpu')

@cudaq.kernel
def bell_kernel():
    q = cudaq.qvector(2)
    h(q[0])
    x.ctrl(q[0], q[1])
    mz(q)

@cudaq.kernel
def ghz3_kernel():
    q = cudaq.qvector(3)
    h(q[0])
    x.ctrl(q[0], q[1])
    x.ctrl(q[0], q[2])
    mz(q)

cudaq_results = {}
for circuit_id, kernel in (('bell', bell_kernel), ('ghz3', ghz3_kernel)):
    seed = seed_for('cudaq_sim', circuit_id)
    cudaq.set_random_seed(seed)
    first = cudaq.sample(kernel, shots_count=SHOTS)
    counts_a = normalize_counts(dict(first.items()))
    cudaq.set_random_seed(seed)
    second = cudaq.sample(kernel, shots_count=SHOTS)
    counts_b = normalize_counts(dict(second.items()))
    assert counts_a == counts_b, f'CUDA-Q no fue reproducible para {circuit_id}'
    assert validate_ideal_counts(counts_a, circuit_id), f'CUDA-Q viola ground truth para {circuit_id}: {counts_a}'
    assert sum(counts_a.values()) == SHOTS
    cudaq_results[circuit_id] = {'seed': seed, 'shots': SHOTS, 'counts': counts_a, 'repeatable': True, 'ground_truth': 'PASS'}
print(json.dumps(cudaq_results, indent=2))

## 5. Modelo de ruido de referencia en Aer (diagnóstico, no gate de modo ideal)

Este bloque verifica que el modelo de ruido puede ejecutarse y genera una distribución de referencia. Los resultados ruidosos **no** se someten al invariante ideal `00/11` o `000/111`, porque los errores pueden producir otros estados.

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 1), ['h'])
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.02, 2), ['cx'])
noisy_backend = AerSimulator(method='density_matrix', device='CPU', noise_model=noise_model)
noisy_counts = normalize_counts(noisy_backend.run(qiskit_circuit('bell'), shots=SHOTS, seed_simulator=seed_for('qiskit_aer', 'bell')).result().get_counts())
assert sum(noisy_counts.values()) == SHOTS
print('Modelo de ruido ejecutado en CPU. Conteos de referencia:')
print(json.dumps(noisy_counts, indent=2))

## 6. Resumen y reporte local

El archivo de salida sólo guarda métricas sintéticas de esta prevalidación local. No contiene credenciales, claves ni datos de proveedor. No actualiza el índice comparativo y no envía información a ningún servicio.

In [ ]:
report = {
    'schema': 'libeqg-phase-c-cpu-report-v1',
    'created_utc_epoch': int(time.time()),
    'python': platform.python_version(),
    'base_seed': BASE_SEED,
    'shots_per_circuit': SHOTS,
    'qiskit_version': qiskit.__version__,
    'qiskit_aer_version': qiskit_aer.__version__,
    'cudaq_version': getattr(cudaq, '__version__', 'unknown'),
    'cudaq_target': 'qpp-cpu',
    'qiskit_aer': aer_results,
    'cudaq_cpu': cudaq_results,
    'aer_noise_diagnostic': {'shots': SHOTS, 'counts': noisy_counts},
    'dwave_leap_beta': {'status': 'DISABLED_BY_DEFAULT', 'executed': False},
    'overall': 'PASS' if all(x['ground_truth'] == 'PASS' for x in list(aer_results.values()) + list(cudaq_results.values())) else 'FAIL',
}
report_path = REPO_ROOT / 'phase_c_cpu_report.json'
report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print('Resultado global:', report['overall'])
print('Reporte:', report_path)
print(json.dumps({k: report[k] for k in ('overall', 'qiskit_version', 'qiskit_aer_version', 'cudaq_version', 'cudaq_target', 'dwave_leap_beta')}, indent=2))
assert report['overall'] == 'PASS', 'Prevalidación local fallida: revisa las celdas con errores antes de usar hardware real.'

## 7. Ejecutar también las pruebas del paquete LIBEQG

Esta celda comprueba el código del repositorio desde su raíz. El resultado de las pruebas opcionales puede incluir `SKIPPED` si no se instalaron paquetes fuera de este notebook; para declarar Aer y CUDA-Q validados, ambas celdas de simulación anteriores deben haber terminado sin errores.

In [ ]:
pytest_result = subprocess.run([sys.executable, '-m', 'pytest', '-v'], cwd=REPO_ROOT, text=True, capture_output=True)
print(pytest_result.stdout)
if pytest_result.stderr:
    print(pytest_result.stderr)
print('Código de salida pytest:', pytest_result.returncode)
if pytest_result.returncode != 0:
    raise RuntimeError('Falló la suite de LIBEQG; no habilites todavía la ejecución contra una QPU real.')